# 10 — Inference, Model Serialization, Reload, and Reproducible Predictions

A model that works only inside the training notebook is not deployable. End-to-end engineering requires a stable inference path.

The inference contract is:

`raw image → same preprocessing → model → logits → probabilities/decision`

Serialization must preserve learned parameters so reloaded predictions match the original model.


In [1]:
from pathlib import Path
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / 'mnist.npz'
MNIST_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
if not MNIST_PATH.exists():
    print('Downloading official MNIST archive...')
    urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
with np.load(MNIST_PATH) as data:
    x_train_raw, y_train_raw = data['x_train'], data['y_train']
    x_test_raw, y_test_raw = data['x_test'], data['y_test']
print('raw train:', x_train_raw.shape, y_train_raw.shape)
print('raw test :', x_test_raw.shape, y_test_raw.shape)


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    ids = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        ids.extend(rng.choice(candidates, size=per_class, replace=False))
    ids = np.array(ids)
    rng.shuffle(ids)
    return x[ids], y[ids]

x_train, y_train = balanced_subset(x_train_raw, y_train_raw, 500)
x_test, y_test = balanced_subset(x_test_raw, y_test_raw, 100)
X_train = x_train.reshape(len(x_train), -1).astype('float32') / 255.0
X_test = x_test.reshape(len(x_test), -1).astype('float32') / 255.0
print('teaching train:', X_train.shape, y_train.shape)
print('teaching test :', X_test.shape, y_test.shape)
print('pixel range   :', float(X_train.min()), 'to', float(X_train.max()))


raw train: (60000, 28, 28) (60000,)
raw test : (10000, 28, 28) (10000,)
teaching train: (5000, 784) (5000,)
teaching test : (1000, 784) (1000,)
pixel range   : 0.0 to 1.0


In [2]:
import tensorflow as tf
tf.random.set_seed(SEED)
model=tf.keras.Sequential([tf.keras.layers.Input((784,)),tf.keras.layers.Dense(64,activation='relu'),tf.keras.layers.Dense(10)])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
model.fit(X_train,y_train,epochs=4,batch_size=128,verbose=0)
MODEL_DIR=ROOT/'artifacts'; MODEL_DIR.mkdir(exist_ok=True); path=MODEL_DIR/'mnist_ann_tensorflow.keras'
model.save(path)
reloaded=tf.keras.models.load_model(path)
p1=tf.nn.softmax(model(X_test[:5],training=False),axis=1); p2=tf.nn.softmax(reloaded(X_test[:5],training=False),axis=1)
print('saved:',path); print('max reload difference:', float(tf.reduce_max(tf.abs(p1-p2)))); print('predictions:',tf.argmax(p2,axis=1).numpy().tolist())


2026-09-07 12:01:32.791655: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-07 12:01:32.818874: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


saved: /home/runner/work/awesome-deep-learning-resource/awesome-deep-learning-resource/artifacts/mnist_ann_tensorflow.keras
max reload difference: 0.0
predictions: [1, 2, 2, 0, 4]


## Why reload verification matters

Saving without reloading only tests the write path. A production-quality workflow must load the artifact into a fresh model object and verify numerical equivalence on known samples. This catches architecture/version/path mistakes before deployment.

## Business relevance

The serialized artifact is what moves through model registries, CI/CD, staging, and production. Governance operates on artifacts and reproducible inference behavior—not on a notebook's in-memory Python object.

### What belongs with the model artifact?

A serious deployment also records the preprocessing contract, feature order, framework/runtime version, model architecture, label mapping, training-data reference, evaluation metrics, and artifact checksum. Without that metadata, a weight file is not a reproducible model package.

### Inference safety check

Reload verification is deliberately performed on the same known samples. If the predictions differ after reload, stop before deployment. Possible causes include incompatible framework versions, a changed architecture, missing preprocessing, incorrect device/dtype handling, or a damaged artifact.
